# Machine Learning 2

# Recommender Systems

## Grading and Penalties

Each task has a specific "cost" (indicated in parentheses next to the task). The maximum allowable score for a task is **10** points.

Ineffective code implementation may negatively impact the score.

## About the Task

In this paper, we'll be solving a music recommendation problem. Our goal is to develop a model that, for each user, will return a set of tracks most similar to those they've already listened to. In the first part, we'll explore a memory-based approach and a latent variable model. These aren't particularly powerful methods, but they allow for near-instantaneous predictions. Then, in the second part, we'll address the fact that the dataset contains a huge number of tracks and use the results of previously built fast models to narrow the candidate list to a manageable number. We'll then rank the candidates using a stronger, but slightly slower, model and select the best options.

Let's get started!

All templates below can be rewritten to suit your needs.

In [ ]:
from sklearn.preprocessing import LabelEncoder

import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from typing import Callable, List

import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as scs

In [ ]:
ratings = pd.read_csv('music_dataset.csv')
ratings.head()

,userId,trackId
0,0,14
1,0,95
2,0,219
3,0,220
4,0,404


In [ ]:
tracks_info = pd.read_csv('tracks_info.csv')
tracks_info.head()

,id,name,artists
0,0,What There Is,['a-ha']
1,1,I'll Play The Blues For You,['Albert King']
2,2,Breaking Up Somebody's Home,['Albert King']
3,3,Imma Be,['Black Eyed Peas']
4,4,Boom Boom Pow,['Black Eyed Peas']


To evaluate the quality of recommendations, we will use the metric $MAP@k$.

$$
MAP@k = \frac{1}{N} \sum_{u = 1}^N AP_u@k
$$
$$
AP_u@k = \frac{1}{\min(k, n_u)} \sum_{i=1}^k r_u(i) p_u@i
$$
$$p_u@k = \dfrac{1}{k}\sum_{j=1}^k r_u(j)$$


* $N$ - number of users.
* $n_u$ - number of relevant tracks of user $u$ in the test interval.
* $r_u(i)$ - binary value: whether the track at position $i$ is relevant.

**Task 1 (0.5 points).** Implement the $MAP@k$ metric.

In [ ]:
def mapk(relevant: List[List[int]], predicted: List[List[int]], k: int = 20):
    # your code here

In [ ]:
relevant = [
    [1, 7, 6, 2, 8],
    [1, 5, 4, 8],
    [8, 2, 5]
]

pred = [
    [8, 1, 5, 0, 7, 2, 9, 4],
    [0, 1, 8, 5, 3, 4, 7, 9],
    [9, 2, 0, 6, 8, 5, 3, 7]
]

assert round(mapk(relevant, pred, k=5), 4) == 0.4331

We will divide the data into training and test data so that the test dataset includes the last 50 tracks of each user.

In [ ]:
def train_test_split(ratings):
    train_ratings, test_ratings = [], []
    num_test_samples = 50

    # getting train samples
    for userId, user_data in tqdm(ratings.groupby('userId')):
        train_ratings += [user_data[:-num_test_samples]]

    train_ratings = pd.concat(train_ratings).reset_index(drop=True)
    all_train_items = train_ratings['trackId'].unique()

    # getting train samples
    # we drop all tracks that are not presented it the training samples,
    # because we won't be able to learn representations for them
    for userId, user_data in tqdm(ratings.groupby('userId')):
        test_items = user_data[-num_test_samples:]
        test_items = test_items[np.isin(test_items['trackId'], all_train_items)]
        test_ratings += [test_items]

    test_ratings = pd.concat(test_ratings).reset_index(drop=True)

    return train_ratings, test_ratings

In [ ]:
train_ratings, test_ratings = train_test_split(ratings)

  0%|          | 0/241 [00:00<?, ?it/s]

  0%|          | 0/241 [00:00<?, ?it/s]

Let's clean up the track information table and encode the track IDs so that they match their serial number.

In [ ]:
redundant_rows = np.where(~np.isin(tracks_info['id'], train_ratings['trackId'].unique()))[0]
tracks_info.drop(redundant_rows, inplace=True)
tracks_info = tracks_info.reset_index(drop=True)

In [ ]:
def ids_encoder(ratings):
    users = sorted(ratings['userId'].unique())
    items = sorted(ratings['trackId'].unique())

    # create users and items encoders
    uencoder = LabelEncoder()
    iencoder = LabelEncoder()

    # fit users and items ids to the corresponding encoder
    uencoder.fit(users)
    iencoder.fit(items)

    return uencoder, iencoder

In [ ]:
uencoder, iencoder = ids_encoder(train_ratings)
train_ratings['trackId'] = iencoder.transform(train_ratings['trackId'].tolist())
test_ratings['trackId'] = iencoder.transform(test_ratings['trackId'].tolist())
tracks_info['id'] = iencoder.transform(tracks_info['id'].tolist())

In [ ]:
train_ratings.head()

,userId,trackId
0,0,14
1,0,95
2,0,219
3,0,220
4,0,404


In [ ]:
test_ratings.head()

,userId,trackId
0,0,57582
1,0,57802
2,0,57957
3,0,58174
4,0,59168


Let's collect all relevant tracks for each user into a list.

In [ ]:
test_relevant = []
test_users = []
for user_id, user_data in test_ratings.groupby('userId'):
    test_relevant += [user_data['trackId'].tolist()]
    test_users.append(user_id)

**Task 2 (0.5 points).** Implement the `get_test_recommendations` method in the `BaseModel` class. It takes the `k` parameter as input and returns an array of the `k` most suitable tracks for each user. Don't forget to remove already played tracks from the recommendations.

In [ ]:
class BaseModel:
    def __init__(self, ratings: pd.DataFrame):
        self.ratings = ratings
        self.n_users = len(np.unique(self.ratings['userId']))
        self.n_items = len(np.unique(self.ratings['trackId']))

        self.R = np.zeros((self.n_users, self.n_items))
        self.R[self.ratings['userId'], self.ratings['trackId']] = 1.

    def recommend(self, uid: int):
        """
        param uid: int - user's id
        return: [n_items] - vector of recommended items sorted by their scores in descending order
        """
        raise NotImplementedError

    def remove_train_items(self, preds: List[List[int]], k: int):
        """
        param preds: [n_users, n_items] - recommended items for each user
        param k: int
        return: np.array [n_users, k] - recommended items without training examples
        """
        new_preds = np.zeros((len(preds), k), dtype=int)
        for user_id, user_data in self.ratings.groupby('userId'):
            user_preds = preds[user_id]
            new_preds[user_id] = user_preds[~np.in1d(user_preds, user_data['trackId'])][:k]

        return new_preds

    def get_test_recommendations(self, k: int):
        test_preds = []

        # your code here

        return test_preds[test_users]

### Part 1. Collaborative Filtering (User2User)

The idea: to select tracks that a user will like, you can collect several similar users (neighbors) and see what tracks they listen to. After that, all that remains is to aggregate the tracks of these users and select the most popular ones. Accordingly, the problem consists of two parts: choosing a similarity function for two users and an aggregation method.

We will use the Jaccard measure as the similarity function:

$$ s(u, v) = \frac{|I_u \cap I_v|}{|I_u \cup I_v|} $$

In all formulas,
* $I_u$ is the set of tracks listened to by user $u$.
* $r_{ui}$ is whether user $u$ has listened to track $i$ (0 or 1).

We define the neighbor set as N(u) = v in U minus u s(u, v) > alpha, where alpha is a hyperparameter.

For aggregation, we will use the following formula.
$$
\hat{r}_{ui} = \frac{\sum_{v \in N(u)} s(u, v) r_{vi}}{\sum_{v \in N(u)} |s(u, v)|}
$$

**Task 3.2 (0.5 points).** Implement a function for calculating the Jaccard score.

The function takes a rating matrix and a rating vector for user $u$ and returns a vector with the similarity values ​​of user $u$ to all users. Try to write optimized code; inefficient implementation may result in a deduction in your score.

In [ ]:
def jaccard(ratings: np.array, user_vector: np.array) -> np.array:
    # your code here
    pass

**Task 4 (1 point).** Implement the `similarity` and `recommend` methods of the `User2User` class. `recommend` returns track indices sorted in descending order of predicted ratings. The value of the `alpha` parameter can be changed as needed to ensure it is reasonable.

In [ ]:
class User2User(BaseModel):
    def __init__(self, ratings):
        super().__init__(ratings)

        self.similarity_func = jaccard
        self.alpha = 0.02

    def similarity(self, user_vector: np.array):
        """
        user_vector: [n_items]
        """
        # your code here
        pass

    def recommend(self, uid: int):
        # your code here
        pass

**Task 5 (0.5 points).** Plot a graph of the $MAP@k$ values ​​as a function of $k$ for a recommendation based on the Jaccard measure. Compare it to recommendations for the most popular tracks and random ones. Which of the three recommendation methods was the best?

In [ ]:
# your code here

**Task 5.1 (1 point).** As you may have noticed, the rating matrix is ​​very sparse, but we're working with it as if it were a regular matrix; that's not appropriate. Rewrite the code so that all methods can work with sparse matrices and compare the performance of this approach with the original one.

In [ ]:
# your code here

We can see how well the model recommends tracks. To do this, we compare tracks already listened to with those recommended and relevant to a random user. How well did you do?

In [ ]:
user_id = np.random.randint(0, model.n_users)

In [ ]:
listened_tracks = train_ratings[train_ratings.userId == user_id].trackId[:15]

print('Already listened tracks:')

tracks_info.loc[listened_tracks][['name', 'artists']]

Already listened tracks:


,name,artists
300,All Star,['Smash Mouth']
831,Tokyo Drift (Fast & Furious),['Teriyaki Boyz']
7272,Around the World (La La La La La),['A Touch Of Class']
8119,Still D.R.E.,['Dr. Dre']
9366,The X-Files,['The X Project']
10707,End Credits,['Hans Zimmer']
16623,Dum Dee Dum,['Keys N Krates']
16911,Mi Mi Mi,['Selene RMX']
18237,Turn Down for What,"['DJ Snake', 'Lil Jon']"
19333,Turn Down for What,"['DJ Snake', 'Lil Jon', 'Juicy J', '2 Chainz',..."


In [ ]:
preds = model.get_test_recommendations(15)

print('Predicted tracks:')

tracks_info.loc[preds[user_id]][['name', 'artists']]

  0%|          | 0/241 [00:00<?, ?it/s]

Predicted tracks:


,name,artists
2814,Numb,['Linkin Park']
24500,Way Down We Go,['KALEO']
1073,Smells Like Teen Spirit,['Nirvana']
805,Zombie,['The Cranberries']
1019,It's My Life,['Bon Jovi']
11493,The Show Must Go On,['Queen']
7533,Highway to Hell,['AC/DC']
49577,Кукла колдуна,['Король и Шут']
18459,Take Me To Church,['Hozier']
8263,Shape Of My Heart,['Sting']


In [ ]:
test_tracks = test_ratings[test_ratings.userId == user_id].trackId[:15]

print('Test-time tracks:')

tracks_info.loc[test_tracks][['name', 'artists']]

Test-time tracks:


,name,artists
65569,Little Do You Know Beat Cry,['Yagih Mael']
65897,Life Goes On,['Oliver Tree']
65904,Gdzie jest biały węgorz ? (Zejście),['Cypis']
65918,Him & Her,['Pacino']
65948,I WANT YOU BACK,['Dazel Ukuto']
65967,Levan Polka,"['Dance', 'Tendência', 'Baila']"
66260,Levan Polkka,['Dj Mix Urbano']
66261,Kulikitaka Challenge,['Dj Mix Urbano']
66299,In Da Getto,"['J. Balvin', 'Skrillex']"
66314,Entrenamiento en el Gym,['Gimnasio de motivación']


### Part 2. Latent Variable Model: ALS

In this section, we'll explore a recommendation method with latent variables.
Idea: we'll predict ratings using the formula
$$
\hat{r}_{ui} = \langle p_u, q_u \rangle,
$$
$p_u \in \mathbb{R}^d$ and $q_i \in \mathbb{R}^d$ are the latent vectors of user $u$ and object $i$, respectively.

We'll optimize the MSE between the user's true rating and the predicted one with regularization
$$
L = \sum_{(u, i) \in R} (\hat{r}_{ui} - r_{ui})^2 + \lambda \left(\sum_{u \in U} \|p_u\|^2 + \sum_{i \in I} \|q_i\|^2\right)
$$

__P. S.__ Note that the described model is designed to work only with explicit information. In our case, the model will always be required to return 1, since we only calculate the error for pairs for which we received feedback. Therefore, it might be logical to think that the problem statement is meaningless. However, in practice, it turns out that due to the random initialization of the $P$ and $Q$ matrices, the trained vectors for all tracks and users at the end of training are different. Therefore, the model is still not without merit.

__P. P. S.__ For more intelligent work with implicit information, the iALS method was proposed; its description can be found in the lecture. A bonus for its implementation is below.

**Task 6 (0.5 points).** The lecture discussed two approaches to parameter optimization. This can be done using standard stochastic gradient descent, or it can be done by updating the $P, Q$ matrices one by one, resulting in the Alternating Least Squares (ALS) method. Derive parameter update formulas for both methods.

**SGD:**

Answer

**ALS:**

Answer

**Task 7 (1 points).** Implement parameter optimization methods for both algorithms.

In [ ]:
class LatentFactorModel(BaseModel):
    def __init__(self, ratings, dim=128, mode='sgd'):
        super().__init__(ratings)
        self.dim = dim

        assert mode in ['sgd', 'als']
        self.mode = mode

        self.P = np.random.normal(size=(self.n_users, dim))
        self.Q = np.random.normal(size=(self.n_items, dim))

        self.lr = 0.0003
        self.lamb = 0.01

    def fit(self, num_iters=5):
        for epoch in tqdm(range(num_iters)):

            if self.mode == 'sgd':
                # your code here
                pass

            elif self.mode == 'als':
                # your code here
                pass

    def recommend(self, uid):
        pred_rating = self.P[uid] @ self.Q.T

        return np.argsort(pred_rating)[::-1]

**Task 8 (1 point).** For both algorithms, find the optimal values ​​of the latent space dimension $d$ and the prediction size $k$. How does the prediction quality change with the number of training iterations? Plot the corresponding graphs, compare them with the random approach and User2User, and draw conclusions. Which algorithm do you think is more suitable for this problem and why?

__P.S.__ At least one of the training methods should yield better results than the User2User approach.

__P.S.__ The SGD method is prone to overfitting, so when selecting parameters, it is useful to look at the error and the optimized metric values ​​on the training dataset. You can also change the initialization and other parameters, except for the architecture, to suit your needs.

In [ ]:
# your code here

If you've achieved sufficiently good quality, then by optimizing the Q parameters, similar tracks now have similar vectors. Therefore, for any track, we can find the closest vectors in the latent space and manually check the model's learning rate.

In [ ]:
example_trackId = tracks_info[tracks_info.name == 'Выхода нет'].iloc[0].id

preds = model.Q @ model.Q[example_trackId]
preds = preds / np.sqrt((model.Q**2).sum(axis=1) + 1e-8)

track_idxs = preds.argsort()[::-1][:20]

In [ ]:
similar_tracks = tracks_info.loc[track_idxs][['name', 'artists']]
similar_tracks['similarity'] = preds[track_idxs] / np.linalg.norm(model.Q[example_trackId])
similar_tracks

,name,artists,similarity
5512,Выхода нет,['Сплин'],1.000000
5517,Варвара,['Би-2'],0.649796
17328,Я хочу быть с тобой,['Nautilus Pompilius'],0.646846
2058,Последний герой,['КИНО'],0.640997
5872,Я свободен,['Кипелов'],0.606749
2060,Хочу перемен,['КИНО'],0.603231
5515,Романс,['Сплин'],0.590318
24284,Как на войне,['Агата Кристи'],0.586219
4463,Holiday,['Green Day'],0.576644
2179,Восьмиклассница,['КИНО'],0.570639


**Bonus (1 points)**

Build an iALS model and compare its performance with ALS and SGD training.

In [ ]:
# your code here: (￣▽￣)/♫•*¨*•.¸¸♪

### Part 3. Second Level of Recommendations.

Above, we built simple models that don't perform very well, but are very fast. We'll use them to select a number of the most promising tracks, which we can then rank using a more complex model (in our case, [CatBoost](https://catboost.ai/en/docs/concepts/python-reference_catboost)).

**Task 9 (0.5 point).**

For each user, take the top 100 recommended tracks from the LFM model (ALS or SGD, your choice) and the top 100 from the User2User model. These will be our candidates, which we will then rank using boosting.

In [ ]:
# your code here

**Task 10 (1 points).**

Prepare a dataset for training the ranking model. It should consist of pairs: object, target variable. An object is a pair (user, item) and any additional features based on them. We suggest creating the following set of features, but you can add your own if you find them reasonable:
1) User ID
1) Track ID
1) Cosine distance between the LFM embeddings of the corresponding user and track
1) Average Jaccard measure between this user and the others in the User2User model
1) Proportion of users who have listened to this track (taken from the training set in Part 1)
1) Number of tracks listened to by the user (taken from the training set in Part 1)

As the target variable, we'll use the binary label "whether the track was in the user's last 50 listens."

Split the resulting set into training and test in a 3:2 ratio so that the proportions of positive and negative examples in both subsets are equal.

In [ ]:
# your code here

As mentioned earlier, we'll use the [CatBoost](https://www.youtube.com/watch?v=dQw4w9WgXcQ) library to build the ranking model.

To transform the dataset into a convenient format, it's useful to use the [`Pool`](https://catboost.ai/en/docs/concepts/python-reference_pool) method.

In [ ]:
import catboost

# group_id == user_id here
train_pool = catboost.Pool(X_train, y_train, group_id=train_group_id)
test_pool = catboost.Pool(X_test, y_test, group_id=test_group_id)

**Task 11 (0.5 point)**

Train the CatBoostClassifier. Use it to make predictions on the test set and calculate MAP@20. Compare these predictions with the User2User and LFM models. Keep in mind that for a fair comparison, we need to recalculate the model predictions on our new test set. Did you improve your results?

In [ ]:
# your code here

**Task 12 (0.5 point)**

Train `CatBoostRanker` by selecting a suitable ranking function from those discussed in the lecture. Follow the same steps as with `CatBoostClassifier` and compare the results.

In [ ]:
# your code here